# cBAD — LightlyTrain distillation on Colab

Downloads [SSamDav/icdar-2019-competition-cbad](https://huggingface.co/datasets/SSamDav/icdar-2019-competition-cbad) directly from Hugging Face, converts its three Parquet splits into image and PAGE-XML trees on the Colab SSD, then distils a randomly initialized **YOLOv8s YAML** student from `dinov3/vitb16`. LightlyTrain receives only the image tree; XML is preserved locally but never passed to training.

The run has a hard safety gate before the first training batch: it reports the full student size, the wrapper-visible size, and — most importantly — the number of student parameters actually present in the optimizer. If that last value is below 10M, training raises `PARAMETER_GUARD_FAILED`, syncs the diagnostic report to Drive, and stops before spending GPU-hours.

Nothing has to be uploaded to Drive. Drive is used only for checkpoints, logs, diagnostics, and the final export:

    MyDrive/Thesis/
      80_models/lightly_train/

The 4.7 GB source dataset and its converted images/XML live only on the Colab SSD. `last.ckpt`, logs, parameter diagnostics, and periodic exports are copied to Drive every 25 epochs; loss is printed and stored every 100 epochs. A restarted runtime resumes the model from Drive and downloads the dataset again.

References: [LightlyTrain distillation](https://docs.lightly.ai/train/stable/pretrain_distill/methods/distillation.html), [data input and output layout](https://docs.lightly.ai/train/stable/train/index.html).

## 1. Config

In [ ]:
import os, shutil, subprocess, sys, time
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules or bool(os.environ.get("COLAB_RELEASE_TAG"))

# ----- edit only this path if your Drive layout differs -------------------
DRIVE_ROOT = Path("/content/drive/MyDrive/Thesis")
WORK_ROOT = Path("/content/work")
# --------------------------------------------------------------------------

HF_DATASET = "SSamDav/icdar-2019-competition-cbad"
HF_CACHE = Path("/content/huggingface-cache")
CBAD_ROOT = WORK_ROOT / "cbad"
CBAD_IMAGES = CBAD_ROOT / "images"
CBAD_GT = CBAD_ROOT / "gt"
EXPECTED_SPLIT_ROWS = {"train": 755, "val": 755, "test": 1511}
OUT_DIR = WORK_ROOT / "lightly_train/cbad_yolov8s_dinov3"
DRIVE_RUN_DIR = DRIVE_ROOT / "80_models/lightly_train/cbad_yolov8s_dinov3"
DRIVE_EXPORT = DRIVE_RUN_DIR / "yolov8s_cbad_distilled_backbone.pt"

MODEL = "ultralytics/yolov8s.yaml"   # YAML only: no COCO initialization
TEACHER = "dinov3/vitb16"
METHOD = "distillation"              # deliberately not dino
BATCH_SIZE = 128
EPOCHS = 3000
MIN_STUDENT_PARAMS = 10_000_000
LOSS_EVERY = 100
CHECKPOINT_EVERY = 25

assert MODEL.endswith(".yaml"), "Refusing to start: the student must be a .yaml, not COCO .pt weights"
assert METHOD == "distillation", "This notebook is for distillation, not DINO pretraining"
print("colab runtime:", IN_COLAB)
print("dataset:      ", HF_DATASET)
print("local data:   ", CBAD_IMAGES)
print("local output: ", OUT_DIR)
print("Drive export: ", DRIVE_EXPORT)

## 2. Mount Drive and check the GPU

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

assert DRIVE_ROOT.is_dir(), f"{DRIVE_ROOT} not found"
DRIVE_RUN_DIR.mkdir(parents=True, exist_ok=True)
subprocess.run(["nvidia-smi"], check=False)

## 3. Install dependencies

`ultralytics` is installed alongside the requested `lightly-train` package because YOLO support is an optional LightlyTrain integration. `datasets` reads the Hugging Face Parquet shards.

In [ ]:
%pip install -q lightly-train ultralytics datasets

## 4. Download and convert cBAD on the local SSD

The repository contains 4.7 GB of Parquet shards with 3,021 records. They are streamed directly from Hugging Face and converted to:

    /content/work/cbad/
      images/{train,val,test}/<filename>.<original-format>
      gt/{train,val,test}/<filename>.xml

The misleading source column name `alto` contains PAGE XML (`PcGts`), not ALTO. A completion marker avoids re-downloading when this cell is re-run in the same Colab runtime. The dataset itself is never written to Drive.

In [ ]:
import json
from io import BytesIO

os.environ.setdefault("HF_HOME", str(HF_CACHE))

import fsspec
from datasets import Image as HFImage, load_dataset
from PIL import Image as PILImage
from tqdm.auto import tqdm

DATASET_MARKER = CBAD_ROOT / "conversion_complete.json"
IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".ppm", ".bmp", ".pgm", ".tif", ".tiff", ".webp"}
FORMAT_SUFFIX = {
    "JPEG": ".jpg", "PNG": ".png", "TIFF": ".tif",
    "BMP": ".bmp", "PPM": ".ppm", "WEBP": ".webp",
}

def converted_counts():
    counts = {}
    for split in EXPECTED_SPLIT_ROWS:
        n_images = sum(1 for p in (CBAD_IMAGES / split).glob("*")
                       if p.is_file() and p.suffix.lower() in IMAGE_SUFFIXES)
        n_xml = sum(1 for p in (CBAD_GT / split).glob("*.xml") if p.is_file())
        counts[split] = {"images": n_images, "xml": n_xml}
    return counts

def conversion_is_complete():
    if not DATASET_MARKER.is_file():
        return False
    try:
        marker = json.loads(DATASET_MARKER.read_text())
    except (OSError, json.JSONDecodeError):
        return False
    return (marker.get("dataset") == HF_DATASET and
            all(marker.get("counts", {}).get(split) == {"images": expected, "xml": expected}
                for split, expected in EXPECTED_SPLIT_ROWS.items()) and
            converted_counts() == marker["counts"])

def raw_image_bytes(image_record):
    payload = image_record.get("bytes")
    source_path = image_record.get("path")
    if payload is None:
        assert source_path, "Image record contains neither bytes nor path"
        with fsspec.open(source_path, "rb") as f:
            payload = f.read()
    suffix = Path(source_path or "").suffix.lower()
    if suffix not in IMAGE_SUFFIXES:
        with PILImage.open(BytesIO(payload)) as image:
            suffix = FORMAT_SUFFIX.get(image.format or "")
    assert suffix in IMAGE_SUFFIXES, f"Unsupported image format for {source_path}"
    return payload, suffix

if conversion_is_complete():
    print("cBAD is already complete in this runtime; skipping download.")
else:
    CBAD_IMAGES.mkdir(parents=True, exist_ok=True)
    CBAD_GT.mkdir(parents=True, exist_ok=True)
    streams = load_dataset(HF_DATASET, streaming=True, cache_dir=str(HF_CACHE))
    actual_counts = {}

    for split, expected in EXPECTED_SPLIT_ROWS.items():
        assert split in streams, f"Missing Hugging Face split: {split}"
        image_dir = CBAD_IMAGES / split
        gt_dir = CBAD_GT / split
        image_dir.mkdir(parents=True, exist_ok=True)
        gt_dir.mkdir(parents=True, exist_ok=True)
        stream = streams[split].cast_column("image", HFImage(decode=False))
        seen = set()

        for row in tqdm(stream, total=expected, desc=f"cBAD {split}"):
            filename = str(row["filename"]).strip()
            assert filename and Path(filename).name == filename, f"Unsafe filename: {filename!r}"
            stem = Path(filename).stem
            assert stem not in seen, f"Duplicate filename in {split}: {stem}"
            seen.add(stem)

            payload, suffix = raw_image_bytes(row["image"])
            image_path = image_dir / f"{stem}{suffix}"
            if not image_path.is_file() or image_path.stat().st_size != len(payload):
                image_path.write_bytes(payload)

            page_xml = str(row["alto"])
            assert "<PcGts" in page_xml[:1000], f"Expected PAGE XML for {split}/{stem}"
            xml_path = gt_dir / f"{stem}.xml"
            if not xml_path.is_file() or xml_path.read_text(encoding="utf-8") != page_xml:
                xml_path.write_text(page_xml, encoding="utf-8")

        assert len(seen) == expected, f"{split}: got {len(seen)} rows, expected {expected}"
        actual_counts[split] = {"images": len(seen), "xml": len(seen)}

    verified = converted_counts()
    assert verified == actual_counts, f"Converted file count mismatch: {verified} vs {actual_counts}"
    DATASET_MARKER.write_text(json.dumps({
        "dataset": HF_DATASET,
        "counts": verified,
    }, indent=2) + "\n")

counts = converted_counts()
assert all(counts[s] == {"images": n, "xml": n} for s, n in EXPECTED_SPLIT_ROWS.items())
print("converted:", counts)
print("LightlyTrain input (images only):", CBAD_IMAGES)
print("PAGE XML kept locally at:         ", CBAD_GT)

In [ ]:
import datasets, lightly_train, torch, ultralytics

assert torch.cuda.is_available(), "GPU is not enabled: Runtime -> Change runtime type -> GPU"
print("lightly_train:", getattr(lightly_train, "__version__", "unknown"))
print("datasets:     ", datasets.__version__)
print("ultralytics:  ", ultralytics.__version__)
print("torch:        ", torch.__version__)
print("GPU:          ", torch.cuda.get_device_name(0))

## 5. Restore/sync run state

Only resumable and diagnostic artifacts cross the Drive boundary. The native `exported_last.pt` is also copied under the explicit backbone filename requested above.

In [ ]:
SYNC_FILES = (
    "checkpoints/last.ckpt",
    "exported_models/exported_last.pt",
    "metrics.jsonl",
    "train.log",
    "loss_every_100_epochs.csv",
    "student_parameter_report.json",
)

def copy_changed(src: Path, dst: Path) -> bool:
    if not src.is_file():
        return False
    if dst.is_file() and dst.stat().st_size == src.stat().st_size and dst.stat().st_mtime >= src.stat().st_mtime:
        return False
    dst.parent.mkdir(parents=True, exist_ok=True)
    tmp = dst.with_name(dst.name + ".part")
    shutil.copy2(src, tmp)
    tmp.replace(dst)
    return True

def pull_resume_state():
    n = sum(copy_changed(DRIVE_RUN_DIR / rel, OUT_DIR / rel) for rel in SYNC_FILES)
    print(f"[Drive -> local] {n} file(s)")

def sync_artifacts():
    n = sum(copy_changed(OUT_DIR / rel, DRIVE_RUN_DIR / rel) for rel in SYNC_FILES)
    print(f"[local -> Drive] {n} file(s)")

def publish_export():
    src = OUT_DIR / "exported_models/exported_last.pt"
    assert src.is_file(), f"Expected export was not created: {src}"
    copy_changed(src, DRIVE_EXPORT)
    print(f"exported backbone -> {DRIVE_EXPORT} ({DRIVE_EXPORT.stat().st_size / 2**20:.1f} MiB)")

pull_resume_state()
RESUME = (OUT_DIR / "checkpoints/last.ckpt").is_file()
print("resume interrupted run:", RESUME)

## 6. Preflight: the YAML itself resolves to YOLOv8s

This cheap CPU check must be near 11M, but it is **not** the safety gate: the real gate below runs after LightlyTrain has created the distillation wrapper and optimizer.

In [ ]:
from ultralytics import YOLO

yaml_name = MODEL.split("/", 1)[1]
probe = YOLO(yaml_name)
yaml_params = sum(p.numel() for p in probe.model.parameters())
print(f"YAML architecture parameters: {yaml_params:,} ({yaml_params / 1e6:.2f}M)")
assert yaml_params >= MIN_STUDENT_PARAMS, (
    f"{yaml_name} resolved to only {yaml_params / 1e6:.2f}M parameters; expected YOLOv8s (~11M)"
)
del probe
torch.cuda.empty_cache()

## 7. Install the start guard and 100-epoch loss logger

LightlyTrain's public `callbacks=` argument configures built-ins but does not accept arbitrary callback instances. This cell adds two small callbacks to the callback list while leaving the built-ins intact. The hook is deliberately fail-closed: if the installed LightlyTrain internals change and the student or optimizer cannot be inspected, training stops.

In [ ]:
import csv, json
from collections.abc import Mapping
from pytorch_lightning import Callback
from lightly_train._callbacks import callback_helpers

PARAM_REPORT = OUT_DIR / "student_parameter_report.json"
LOSS_LOG = OUT_DIR / "loss_every_100_epochs.csv"

def unique_parameters(module):
    return {id(p): p for p in module.parameters()}.values()

class StudentParameterGuard(Callback):
    """Count student parameters actually connected to an optimizer, before batch 1."""

    def on_train_start(self, trainer, pl_module):
        if not trainer.is_global_zero:
            return
        embedding = getattr(pl_module, "student_embedding_model", None)
        if embedding is None or not hasattr(embedding, "wrapped_model"):
            raise RuntimeError("PARAMETER_GUARD_FAILED: cannot find student_embedding_model.wrapped_model")
        wrapped = embedding.wrapped_model
        if not hasattr(wrapped, "get_model"):
            raise RuntimeError("PARAMETER_GUARD_FAILED: wrapper has no get_model()")
        package_model = wrapped.get_model()
        raw_model = getattr(package_model, "model", package_model)

        full_params = list(unique_parameters(raw_model))
        wrapper_params = list(unique_parameters(embedding))
        optimizer_ids = {id(p)
                         for optimizer in trainer.optimizers
                         for group in optimizer.param_groups
                         for p in group["params"]}
        if not optimizer_ids:
            raise RuntimeError("PARAMETER_GUARD_FAILED: trainer has no optimizer parameters")

        full_total = sum(p.numel() for p in full_params)
        full_requires_grad = sum(p.numel() for p in full_params if p.requires_grad)
        wrapper_visible = sum(p.numel() for p in wrapper_params if p.requires_grad)
        optimizer_student = sum(p.numel() for p in full_params
                                if p.requires_grad and id(p) in optimizer_ids)

        layers = []
        seq = getattr(raw_model, "model", None)
        if seq is not None:
            for index, layer in enumerate(seq):
                params = list(unique_parameters(layer))
                layers.append({
                    "index": index,
                    "type": type(layer).__name__,
                    "parameters": sum(p.numel() for p in params),
                    "optimizer_parameters": sum(p.numel() for p in params if id(p) in optimizer_ids),
                })

        report = {
            "lightly_train_version": getattr(lightly_train, "__version__", "unknown"),
            "ultralytics_version": ultralytics.__version__,
            "torch_version": torch.__version__,
            "model": MODEL,
            "teacher": TEACHER,
            "minimum_optimizer_student_parameters": MIN_STUDENT_PARAMS,
            "full_student_parameters": full_total,
            "full_student_requires_grad_parameters": full_requires_grad,
            "wrapper_visible_trainable_parameters": wrapper_visible,
            "optimizer_covered_student_parameters": optimizer_student,
            "passes": optimizer_student >= MIN_STUDENT_PARAMS,
            "layers": layers,
        }
        PARAM_REPORT.parent.mkdir(parents=True, exist_ok=True)
        PARAM_REPORT.write_text(json.dumps(report, indent=2) + "\n")

        print("\n" + "=" * 78)
        print("STUDENT PARAMETER INTEGRITY CHECK (after trainer/optimizer start)")
        print(f"full YOLO student:          {full_total:>12,}  ({full_total / 1e6:6.2f}M)")
        print(f"full student requires_grad: {full_requires_grad:>12,}  ({full_requires_grad / 1e6:6.2f}M)")
        print(f"wrapper-visible trainable:  {wrapper_visible:>12,}  ({wrapper_visible / 1e6:6.2f}M)")
        print(f"IN STUDENT OPTIMIZER:        {optimizer_student:>12,}  ({optimizer_student / 1e6:6.2f}M)")
        print(f"required minimum:           {MIN_STUDENT_PARAMS:>12,}  ({MIN_STUDENT_PARAMS / 1e6:6.2f}M)")
        print("=" * 78)

        if optimizer_student < MIN_STUDENT_PARAMS:
            sync_artifacts()
            raise RuntimeError(
                "PARAMETER_GUARD_FAILED: only "
                f"{optimizer_student / 1e6:.2f}M student parameters are connected to the optimizer; "
                f"need at least {MIN_STUDENT_PARAMS / 1e6:.2f}M. Training stopped before the first batch. "
                f"Report: {DRIVE_RUN_DIR / PARAM_REPORT.name}"
            )

class LossAndSync(Callback):
    def __init__(self):
        self.loss_sum = 0.0
        self.loss_count = 0
        self.last_logged_loss = None
        if LOSS_LOG.is_file():
            with LOSS_LOG.open(newline="") as f:
                rows = list(csv.DictReader(f))
            if rows:
                self.last_logged_loss = float(rows[-1]["mean_train_loss"])

    def on_train_epoch_start(self, trainer, pl_module):
        self.loss_sum = 0.0
        self.loss_count = 0

    def on_train_batch_end(self, trainer, pl_module, outputs, batch, batch_idx):
        loss = outputs.get("loss") if isinstance(outputs, Mapping) else outputs
        if isinstance(loss, torch.Tensor) and loss.numel() == 1:
            self.loss_sum += float(loss.detach().cpu())
            self.loss_count += 1

    def on_train_epoch_end(self, trainer, pl_module):
        if not trainer.is_global_zero:
            return
        epoch = trainer.current_epoch + 1
        if epoch % LOSS_EVERY == 0:
            loss = self.loss_sum / max(self.loss_count, 1)
            LOSS_LOG.parent.mkdir(parents=True, exist_ok=True)
            new_file = not LOSS_LOG.exists()
            with LOSS_LOG.open("a", newline="") as f:
                writer = csv.writer(f)
                if new_file:
                    writer.writerow(["epoch", "mean_train_loss", "global_step"])
                writer.writerow([epoch, f"{loss:.8f}", trainer.global_step])
            print(f"\n[LOSS @ epoch {epoch:4d}] mean train_loss = {loss:.6f}")
            if epoch > 500 and self.last_logged_loss is not None:
                delta = loss - self.last_logged_loss
                status = "DOWN" if delta < 0 else "WARNING: NOT DOWN"
                print(f"[post-500 trend] {status}; delta over {LOSS_EVERY} epochs = {delta:+.6f}")
            self.last_logged_loss = loss
        if epoch % CHECKPOINT_EVERY == 0:
            sync_artifacts()

# Preserve the original function across cell re-runs; do not stack callbacks.
if not hasattr(callback_helpers, "_cbad_original_get_callbacks"):
    callback_helpers._cbad_original_get_callbacks = callback_helpers.get_callbacks
original_get_callbacks = callback_helpers._cbad_original_get_callbacks

def get_callbacks_with_cbad_guards(*args, **kwargs):
    callbacks = original_get_callbacks(*args, **kwargs)
    callbacks.extend([StudentParameterGuard(), LossAndSync()])
    return callbacks

callback_helpers.get_callbacks = get_callbacks_with_cbad_guards
print("Installed fail-closed student parameter guard and 100-epoch loss logger.")

## 8. Train

The call below deliberately mirrors the requested configuration. On a fresh run, `overwrite=True` only applies to the dedicated local SSD output directory; Drive is never deleted. On restart, `resume_interrupted=True` restores the optimizer and epoch from `last.ckpt`.

In [ ]:
completed = False
try:
    lightly_train.train(
        out=str(OUT_DIR),
        data=str(CBAD_IMAGES),
        model=MODEL,                         # ultralytics/yolov8s.yaml, never .pt
        method=METHOD,                       # distillation, never dino
        method_args={"teacher": TEACHER},
        batch_size=BATCH_SIZE,
        epochs=EPOCHS,
        resume_interrupted=RESUME,
        overwrite=not RESUME,
        callbacks={
            "model_checkpoint": {
                "every_n_epochs": CHECKPOINT_EVERY,
                "save_last": True,
                "save_top_k": 0,
            },
            "model_export": {"every_n_epochs": LOSS_EVERY},
        },
    )
    completed = True
finally:
    # Also runs after PARAMETER_GUARD_FAILED or a disconnected Colab session.
    sync_artifacts()

if completed:
    publish_export()

## 9. Inspect the 100-epoch loss trace and exported backbone

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

assert DRIVE_EXPORT.is_file(), f"Drive export missing: {DRIVE_EXPORT}"
print("Drive export:", DRIVE_EXPORT)
print(f"size: {DRIVE_EXPORT.stat().st_size / 2**20:.1f} MiB")

if LOSS_LOG.is_file():
    losses = pd.read_csv(LOSS_LOG).drop_duplicates("epoch", keep="last").sort_values("epoch")
    display(losses.tail(20))
    ax = losses.plot(x="epoch", y="mean_train_loss", marker="o", figsize=(9, 4), grid=True, legend=False)
    ax.axvline(500, color="tab:red", linestyle="--", alpha=0.7, label="epoch 500")
    ax.set(title="cBAD distillation loss (logged every 100 epochs)", ylabel="mean train loss")
    ax.legend()
    plt.show()
else:
    print("No 100-epoch loss records yet.")